# EMPIEZA ARA

# Learning Urban Crimes Representation

## Construcción de firmas espacio temporales
Input:  carpetasFGJ_fase2.csv (con macro_categoria y delito_limpio)
Output: firmas_h3.csv — una fila por hexágono H3, con su firma multivista
        h3_metadata.csv — metadatos por hexágono (alcaldía, colonia dominante)
        carpetasFGJ_fase2_h3.csv — dataset con columna h3_id añadida

Firma por hexágono:
  - Mezcla de delitos (proporción por macro_categoría) — 26 dims
  - Perfil horario (proporción por franja de 4 horas) — 6 dims
  - Perfil semanal (proporción por día de semana) — 7 dims
  - Estacionalidad (proporción por trimestre) — 4 dims
  - Intensidad (log del conteo total) — 1 dim
  - Ratio violencia (proporción de delitos con violencia) — 1 dim
  Total: ~45 dimensiones por hexágono

### Paquetes

In [1]:
import pandas as pd
import numpy as np
import h3

### Ejecución

#### Cargar datos

In [3]:

df = pd.read_csv("../data/phases/carpetasFGJ_fase2.csv")
print(f"Registros totales: {len(df):,}")

# Parsear fechas y horas
df['fecha_hecho_dt'] = pd.to_datetime(df['fecha_hecho'], errors='coerce')
df['hora_hecho_dt'] = pd.to_datetime(df['hora_hecho'], format='%H:%M:%S', errors='coerce')

# Filtrar registros válidos para H3 (coordenadas no nulas, fecha no sospechosa, dentro del bbox de CDMX)
mask_coords = df['latitud'].notna() & df['longitud'].notna()
mask_fecha = df['fecha_sospechosa'] == False
mask_bbox = (
    (df['latitud'] >= 19.05) & (df['latitud'] <= 19.60) &
    (df['longitud'] >= -99.40) & (df['longitud'] <= -98.90)
)

df_valid = df[mask_coords & mask_fecha & mask_bbox].copy()
print(f"Registros válidos (coords + fecha + bbox): {len(df_valid):,} ({len(df_valid)/len(df)*100:.1f}%)")
print(f"Descartados: {len(df) - len(df_valid):,}")

C:\Users\adolh\AppData\Local\Temp\ipykernel_10136\3237964835.py:1: DtypeWarning: Columns (0: competencia) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/phases/carpetasFGJ_fase2.csv")


Registros totales: 2,098,743
Registros válidos (coords + fecha + bbox): 1,967,472 (93.7%)
Descartados: 131,271


#### Implementación

In [4]:
# ============================================================================
# PASO 1: Asignar hexágonos H3 (resolución 8 ≈ 460m diámetro)
# ============================================================================
H3_RESOLUTION = 8

print(f"\nAsignando hexágonos H3 (resolución {H3_RESOLUTION})...")
df_valid['h3_id'] = df_valid.apply(
    lambda row: h3.latlng_to_cell(row['latitud'], row['longitud'], H3_RESOLUTION),
    axis=1
)

n_hexagonos = df_valid['h3_id'].nunique()
print(f"Hexágonos únicos: {n_hexagonos:,}")
print(f"Registros por hexágono (mediana): {df_valid.groupby('h3_id').size().median():.0f}")
print(f"Registros por hexágono (media): {df_valid.groupby('h3_id').size().mean():.0f}")

# Distribución de tamaños
sizes = df_valid.groupby('h3_id').size()
print(f"\nDistribución de registros por hexágono:")
print(f"  Min:  {sizes.min()}")
print(f"  P25:  {sizes.quantile(0.25):.0f}")
print(f"  P50:  {sizes.quantile(0.50):.0f}")
print(f"  P75:  {sizes.quantile(0.75):.0f}")
print(f"  P95:  {sizes.quantile(0.95):.0f}")
print(f"  Max:  {sizes.max()}")

# Filtrar hexágonos con muy pocos registros (ruido estadístico)
MIN_REGISTROS = 30
hexagonos_validos = sizes[sizes >= MIN_REGISTROS].index
n_filtrados = n_hexagonos - len(hexagonos_validos)
registros_perdidos = df_valid[~df_valid['h3_id'].isin(hexagonos_validos)].shape[0]

print(f"\nFiltro mínimo ({MIN_REGISTROS} registros):")
print(f"  Hexágonos descartados: {n_filtrados:,} de {n_hexagonos:,}")
print(f"  Registros perdidos: {registros_perdidos:,} ({registros_perdidos/len(df_valid)*100:.2f}%)")
print(f"  Hexágonos finales: {len(hexagonos_validos):,}")

df_h3 = df_valid[df_valid['h3_id'].isin(hexagonos_validos)].copy()


Asignando hexágonos H3 (resolución 8)...
Hexágonos únicos: 1,181
Registros por hexágono (mediana): 1188
Registros por hexágono (media): 1666

Distribución de registros por hexágono:
  Min:  1
  P25:  248
  P50:  1188
  P75:  2436
  P95:  4824
  Max:  21979

Filtro mínimo (30 registros):
  Hexágonos descartados: 120 de 1,181
  Registros perdidos: 888 (0.05%)
  Hexágonos finales: 1,061


In [6]:

# ============================================================================
# PASO 2: Extraer features temporales
# ============================================================================
#print(f"\nExtrayendo features temporales...")

# Hora del día → franja de 4 horas (6 franjas)
# 0: 00-04 (madrugada), 1: 04-08 (amanecer), 2: 08-12 (mañana),
# 3: 12-16 (mediodía), 4: 16-20 (tarde), 5: 20-24 (noche)
df_h3['hora_int'] = df_h3['hora_hecho_dt'].dt.hour
df_h3['franja_horaria'] = pd.cut(
    df_h3['hora_int'],
    bins=[-1, 4, 8, 12, 16, 20, 24],
    labels=['madrugada', 'amanecer', 'manana', 'mediodia', 'tarde', 'noche']
)

# Día de la semana
df_h3['dia_semana'] = df_h3['fecha_hecho_dt'].dt.day_name()

# Trimestre
df_h3['trimestre'] = df_h3['fecha_hecho_dt'].dt.quarter.map({
    1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'
})

# ============================================================================
# PASO 3: Construir firma por hexágono
# ============================================================================
#print(f"Construyendo firmas por hexágono...")

# --- 3a. Mezcla de delitos (proporción por macro_categoría) ---
delito_counts = df_h3.groupby(['h3_id', 'macro_categoria']).size().unstack(fill_value=0)
delito_props = delito_counts.div(delito_counts.sum(axis=1), axis=0)
delito_props.columns = [f'delito_{c.lower().replace(" ", "_")}' for c in delito_props.columns]

# --- 3b. Perfil horario (solo registros con hora válida) ---
df_hora_valida = df_h3[df_h3['hora_hecho_valida'] == True]
hora_counts = df_hora_valida.groupby(['h3_id', 'franja_horaria']).size().unstack(fill_value=0)
# Asegurar que todas las franjas existan
for franja in ['madrugada', 'amanecer', 'manana', 'mediodia', 'tarde', 'noche']:
    if franja not in hora_counts.columns:
        hora_counts[franja] = 0
hora_props = hora_counts.div(hora_counts.sum(axis=1), axis=0)
hora_props.columns = [f'hora_{c}' for c in hora_props.columns]
# Hexágonos sin horas válidas → distribución uniforme
hora_props = hora_props.reindex(delito_props.index).fillna(1/6)

# --- 3c. Perfil semanal ---
dia_counts = df_h3.groupby(['h3_id', 'dia_semana']).size().unstack(fill_value=0)
dias_orden = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
for dia in dias_orden:
    if dia not in dia_counts.columns:
        dia_counts[dia] = 0
dia_counts = dia_counts[dias_orden]
dia_props = dia_counts.div(dia_counts.sum(axis=1), axis=0)
dia_props.columns = [f'dia_{c.lower()}' for c in dia_props.columns]
dia_props = dia_props.reindex(delito_props.index).fillna(1/7)

# --- 3d. Estacionalidad (trimestres) ---
trim_counts = df_h3.groupby(['h3_id', 'trimestre']).size().unstack(fill_value=0)
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    if q not in trim_counts.columns:
        trim_counts[q] = 0
trim_props = trim_counts.div(trim_counts.sum(axis=1), axis=0)
trim_props.columns = [f'trim_{c}' for c in trim_props.columns]
trim_props = trim_props.reindex(delito_props.index).fillna(1/4)

# --- 3e. Intensidad (log del conteo total) ---
conteo_total = df_h3.groupby('h3_id').size()
intensidad = np.log1p(conteo_total).rename('intensidad_log')
intensidad = intensidad.reindex(delito_props.index)

# --- 3f. Ratio de violencia ---
macro_violentas = ['ROBO CON VIOLENCIA', 'HOMICIDIO', 'FEMINICIDIO',
                   'LESIONES INTENCIONALES', 'PRIVACION DE LIBERTAD']
df_h3['es_violento'] = df_h3['macro_categoria'].isin(macro_violentas)
ratio_violencia = df_h3.groupby('h3_id')['es_violento'].mean().rename('ratio_violencia')
ratio_violencia = ratio_violencia.reindex(delito_props.index).fillna(0)

# ============================================================================
# PASO 4: Ensamblar firma completa
# ============================================================================
#print(f"Ensamblando firmas...")

firmas = pd.concat([
    delito_props,      # ~26 dims
    hora_props,        # 6 dims
    dia_props,         # 7 dims
    trim_props,        # 4 dims
    intensidad,        # 1 dim
    ratio_violencia,   # 1 dim
], axis=1)

# Verificar
print(f"\n{'='*80}")
print(f"FIRMA ESPACIO-TEMPORAL — RESUMEN")
print(f"{'='*80}")
print(f"  Hexágonos: {len(firmas):,}")
print(f"  Dimensiones: {firmas.shape[1]}")
print(f"  Nulos totales: {firmas.isna().sum().sum()}")

print(f"\n  Composición de la firma:")
print(f"    Mezcla de delitos:   {len(delito_props.columns)} dims")
print(f"    Perfil horario:      {len(hora_props.columns)} dims")
print(f"    Perfil semanal:      {len(dia_props.columns)} dims")
print(f"    Estacionalidad:      {len(trim_props.columns)} dims")
print(f"    Intensidad:          1 dim")
print(f"    Ratio violencia:     1 dim")
print(f"    TOTAL:               {firmas.shape[1]} dims")

# Estadísticas descriptivas rápidas
print(f"\n  Intensidad (log registros):")
print(f"    Min:  {firmas['intensidad_log'].min():.2f} ({np.expm1(firmas['intensidad_log'].min()):.0f} registros)")
print(f"    P50:  {firmas['intensidad_log'].median():.2f} ({np.expm1(firmas['intensidad_log'].median()):.0f} registros)")
print(f"    Max:  {firmas['intensidad_log'].max():.2f} ({np.expm1(firmas['intensidad_log'].max()):.0f} registros)")

print(f"\n  Ratio de violencia:")
print(f"    Min:  {firmas['ratio_violencia'].min():.3f}")
print(f"    P50:  {firmas['ratio_violencia'].median():.3f}")
print(f"    Max:  {firmas['ratio_violencia'].max():.3f}")

# Top 5 macro-categorías por peso promedio
print(f"\n  Macro-categorías con mayor peso promedio en las firmas:")
delito_cols = [c for c in firmas.columns if c.startswith('delito_')]
promedios = firmas[delito_cols].mean().sort_values(ascending=False)
for col, val in promedios.head(8).items():
    nombre = col.replace('delito_', '').replace('_', ' ').upper()
    print(f"    {nombre:45s} {val:.3f} ({val*100:.1f}%)")



FIRMA ESPACIO-TEMPORAL — RESUMEN
  Hexágonos: 1,061
  Dimensiones: 45
  Nulos totales: 0

  Composición de la firma:
    Mezcla de delitos:   26 dims
    Perfil horario:      6 dims
    Perfil semanal:      7 dims
    Estacionalidad:      4 dims
    Intensidad:          1 dim
    Ratio violencia:     1 dim
    TOTAL:               45 dims

  Intensidad (log registros):
    Min:  3.43 (30 registros)
    P50:  7.30 (1485 registros)
    Max:  10.00 (21979 registros)

  Ratio de violencia:
    Min:  0.025
    P50:  0.139
    Max:  0.435

  Macro-categorías con mayor peso promedio en las firmas:
    ROBO SIN VIOLENCIA                            0.245 (24.5%)
    VIOLENCIA FAMILIAR                            0.168 (16.8%)
    FRAUDE Y DELITOS PATRIMONIALES                0.133 (13.3%)
    ROBO CON VIOLENCIA                            0.108 (10.8%)
    AMENAZAS                                      0.075 (7.5%)
    DANO EN PROPIEDAD                             0.054 (5.4%)
    DELITOS SEXUALE

# TERMINA ARA

# EMPIEZA NICOLE

In [7]:

# ============================================================================
# PASO 5: Metadatos por hexágono (para interpretación en etapa 5.4)
# ============================================================================
print(f"\nGenerando metadatos por hexágono...")

# Alcaldía dominante
alcaldia_dom = df_h3.groupby('h3_id')['alcaldia_hecho'].agg(
    lambda x: x.value_counts().index[0] if len(x) > 0 else 'DESCONOCIDA'
).rename('alcaldia_dominante')

# Colonia dominante
colonia_dom = df_h3.groupby('h3_id')['colonia_hecho'].agg(
    lambda x: x.dropna().value_counts().index[0] if len(x.dropna()) > 0 else 'DESCONOCIDA'
).rename('colonia_dominante')

# Centro del hexágono (para mapeo)
centros = pd.DataFrame(
    [h3.cell_to_latlng(h) for h in firmas.index],
    index=firmas.index,
    columns=['lat_centro', 'lon_centro']
)

# Conteo total
conteo_df = conteo_total.rename('n_registros')

metadata = pd.concat([alcaldia_dom, colonia_dom, centros, conteo_df], axis=1)
metadata = metadata.reindex(firmas.index)

# Distribución por alcaldía
print(f"\n  Hexágonos por alcaldía:")
for alc, count in metadata['alcaldia_dominante'].value_counts().head(16).items():
    print(f"    {alc:30s} {count:>5} hexágonos")



Generando metadatos por hexágono...

  Hexágonos por alcaldía:
    IZTAPALAPA                       143 hexágonos
    TLALPAN                          132 hexágonos
    GUSTAVO A. MADERO                112 hexágonos
    XOCHIMILCO                        86 hexágonos
    ALVARO OBREGON                    84 hexágonos
    TLAHUAC                           65 hexágonos
    COYOACAN                          62 hexágonos
    MILPA ALTA                        57 hexágonos
    MIGUEL HIDALGO                    57 hexágonos
    CUAJIMALPA DE MORELOS             51 hexágonos
    AZCAPOTZALCO                      47 hexágonos
    VENUSTIANO CARRANZA               38 hexágonos
    CUAUHTEMOC                        37 hexágonos
    BENITO JUAREZ                     34 hexágonos
    IZTACALCO                         29 hexágonos
    LA MAGDALENA CONTRERAS            27 hexágonos


In [8]:

# ============================================================================
# PASO 6: Exportar
# ============================================================================
# 6a. Firmas
firmas.index.name = 'h3_id'
firmas.to_csv('../data/auxiliar/firmas_h3.csv', encoding='utf-8-sig')
print(f"\nfirmas_h3.csv ({len(firmas):,} hexágonos × {firmas.shape[1]} dimensiones)")

# 6b. Metadatos
metadata.index.name = 'h3_id'
metadata.to_csv('../data/auxiliar/h3_metadata.csv', encoding='utf-8-sig')
print(f"h3_metadata.csv ({len(metadata):,} hexágonos)")

# 6c. Dataset con h3_id
df_valid['h3_id'] = df_valid.apply(
    lambda row: h3.latlng_to_cell(row['latitud'], row['longitud'], H3_RESOLUTION),
    axis=1
)
df_valid.to_csv('../data/phases/carpetasFGJ_fase2_h3.csv', index=False, encoding='utf-8-sig')
print(f"carpetasFGJ_fase2_h3.csv ({len(df_valid):,} registros)")


firmas_h3.csv (1,061 hexágonos × 45 dimensiones)
h3_metadata.csv (1,061 hexágonos)
carpetasFGJ_fase2_h3.csv (1,967,472 registros)


# TERMINA NICOLE